In [1]:
import json
from safetensors import safe_open
from safetensors.torch import save_file
from pathlib import Path
from tqdm import tqdm

def rename_keys(input_path, output_path):
    tensors = {}
    metadata = None

    # Load tensors and metadata
    with safe_open(input_path, framework="pt", device="cpu") as f:
        metadata = f.metadata()  # preserves all custom metadata
        for key in f.keys():
            assert "grad" not in key, "expected un-renamed file"
            step_idx, param_name = key.split(".", 1)
            new_key = f"{step_idx}.grad.{param_name}"
            tensors[new_key] = f.get_tensor(key)

    # Save with original metadata intact
    save_file(tensors, output_path, metadata=metadata)

def rename_directory(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    safetensors_files = sorted(list(input_dir.glob("*.safetensors")))
    pbar = tqdm(safetensors_files, desc=f"Processing files")
    for filename in pbar:
        pbar.set_postfix_str(f"Processing {filename.name}")
        input_path = filename
        output_path = Path(output_dir) / filename.name
        rename_keys(input_path, output_path)

In [3]:
subset = "algorithm-generated"
model = "qwen3-8b"
input_dir = f"outputs/temps/{model}/{subset}"
output_dir = f"outputs/grads/{model}/reps/{subset}"
rename_directory(input_dir, output_dir)

Processing files: 100%|██████████| 126/126 [00:43<00:00,  2.91it/s, Processing 99.safetensors]


In [ ]:
# if __name__ == "__main__":
#     for subset in ["hand-crafted", "algorithm-generated"]:
#         print(f"Processing subset: {subset}")
#         input_dir = f"outputs/grads/llama-3.1-8b/old_reps/{subset}"
#         output_dir = f"outputs/grads/llama-3.1-8b/reps/{subset}"
#         rename_directory(input_dir, output_dir)

In [ ]:
import pandas as pd
from pathlib import Path

metrics_root = Path("outputs/hidden")

rows = []
for tsv in metrics_root.glob("*/svd/metrics/*/*/k1_*.tsv"):
    parts = tsv.parts  # .../<model>/svd/metrics/<subset>/<config>/k1_<dir>.tsv
    model, subset, config = parts[-5], parts[-3], parts[-2]
    df = pd.read_csv(tsv, sep="\t")
    best = df.loc[df["step_acc"].idxmax()]
    rows.append({
        "model": model, "subset": subset, "config": config,
        "direction": tsv.stem.split("_", 1)[1],  # asc / desc
        "weight": best["weight"],
        "step_acc": best["step_acc"],
        "agent_acc": best["agent_acc"],
    })

# results = (
#     pd.DataFrame(rows)
#     .sort_values("step_acc", ascending=False)
#     .reset_index(drop=True)
# )
# # results

KeyError: 'step_acc'